<a href="https://colab.research.google.com/github/rgak001/AdMob/blob/main/Wan2_1_14B_T2V_GGUF_Free.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Wan2.1-14b Quantized GGUF Models for Text to Video**

- You can use the free T4 GPU to run this. For faster video generation, use higher GPUs.
- Generating a video with 32 frames at 512 by 910 resolution can take up to 28 minutes on the T4 GPU using the Q5 model.
- It's possible to generate up to 24 frames at 480 by 832 resolution for free using the Q6 model. The generation takes about 20 minutes.
- Set `frames` to 1 to generate an image. You can generate high quality images with 720 by 1280 resolution using either the Q5 or Q6 gguf models. The generation takes less than 7 minutes.

In [1]:
# @title Prepare Environment
!pip install torch==2.6.0 torchvision==0.21.0
%cd /content

!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2
!pip install av
!git clone https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg

useQ6 = True # @param {"type":"boolean"}

if useQ6:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q6_K.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q6_K.gguf
else:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q5_0.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q5_0.gguf

!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors

import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import imageio
import subprocess
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode,
    VAELoader,
    KSampler,
    UNETLoader
)

from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_hunyuan import EmptyHunyuanLatentVideo
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM

# unet_loader = UNETLoader()
unet_loader = UnetLoaderGGUF()
# model_sampling = ModelSamplingSD3()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
empty_latent_video = EmptyHunyuanLatentVideo()
ksampler = KSampler()
vae_decode = VAEDecode()
save_webp = SaveAnimatedWEBP()
save_webm = SaveWEBM()

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()

def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"

    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webp(images, filename_prefix, fps, quality=90, lossless=False, method=4, output_dir="/content/ComfyUI/output"):
    """Save images as animated WEBP using imageio."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webp"


    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]


    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'lossless': bool(lossless),
        'method': int(method)
    }

    with imageio.get_writer(
        output_path,
        format='WEBP',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=32, output_dir="/content/ComfyUI/output"):
    """Save images as WEBM using imageio."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"


    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]


    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'codec': str(codec),
        'output_params': ['-crf', str(int(quality))]
    }

    with imageio.get_writer(
        output_path,
        format='FFMPEG',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    """Save single frame as PNG image."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"

    frame = (image.cpu().numpy() * 255).astype(np.uint8)

    Image.fromarray(frame).save(output_path)

    return output_path

def generate_video(
    positive_prompt: str = "a fox moving quickly in a beautiful winter scenery nature trees mountains daytime tracking camera",
    negative_prompt: str = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走",
    width: int = 832,
    height: int = 480,
    seed: int = 82628696717253,
    steps: int = 30,
    cfg_scale: float = 1.0,
    sampler_name: str = "uni_pc",
    scheduler: str = "simple",
    frames: int = 33,
    fps: int = 16,
    output_format: str = "mp4"
):

    with torch.inference_mode():
        print("Loading Text_Encoder...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]

        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        torch.cuda.empty_cache()
        gc.collect()

        empty_latent = empty_latent_video.generate(width, height, frames, 1)[0]

        print("Loading Unet Model...")
        if useQ6:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q6_K.gguf")[0]
        else:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q5_0.gguf")[0]
        # model = model_sampling.patch(model, 8)[0]

        print("Generating video...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive,
            negative=negative,
            latent_image=empty_latent
        )[0]

        del model
        torch.cuda.empty_cache()
        gc.collect()

        print("Loading VAE...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]

        try:
            print("Decoding latents...")
            decoded = vae_decode.decode(vae, sampled)[0]

            del vae
            torch.cuda.empty_cache()
            gc.collect()

            output_path = ""
            if frames == 1:
                print("Single frame detected - saving as PNG image...")
                output_path = save_as_image(decoded[0], "ComfyUI")
                # print(f"Image saved as PNG: {output_path}")

                display(IPImage(filename=output_path))
            else:
                if output_format.lower() == "webm":
                    print("Saving as WEBM...")
                    output_path = save_as_webm(
                        decoded,
                        "ComfyUI",
                        fps=fps,
                        codec="vp9",
                        quality=10
                    )
                elif output_format.lower() == "mp4":
                    print("Saving as MP4...")
                    output_path = save_as_mp4(decoded, "ComfyUI", fps)
                else:
                    raise ValueError(f"Unsupported output format: {output_format}")

                # print(f"Video saved as {output_format.upper()}: {output_path}")

                display_video(output_path)

        except Exception as e:
            print(f"Error during decoding/saving: {str(e)}")
            raise
        finally:
            clear_memory()

def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode

    video_data = open(video_path,'rb').read()

    # Determine MIME type based on file extension
    if video_path.lower().endswith('.mp4'):
        mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'):
        mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'):
        mime_type = "image/webp"
    else:
        mime_type = "video/mp4"  # default

    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()

    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="{mime_type}">
    </video>
    """))

print("✅ Environment Setup Complete!")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 897.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 811.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# @title Batch Text-to-Video Generate (Wan 2.1 14B GGUF Optimized)
import os
import json
import shutil
import glob
import torch
import gc
import imageio
import numpy as np
from PIL import Image
from google.colab import drive

# 1. Mountanje Google Drive-a
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. Konfiguracija putanja
PROJECT_DIR = '/content/drive/MyDrive/Andromeda_Hunter_4K'
RAW_VIDEO_DIR = f"{PROJECT_DIR}/03_raw_video"
JSON_PATH = "/content/scenes_65min.json"

os.makedirs(RAW_VIDEO_DIR, exist_ok=True)

# 3. Učitavanje i čišćenje JSON-a (uklanja razmake iz ključeva)
def clean_keys(obj):
    if isinstance(obj, dict): return {k.strip(): clean_keys(v) for k, v in obj.items()}
    if isinstance(obj, list): return [clean_keys(i) for i in obj]
    if isinstance(obj, str): return obj.strip()
    return obj

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)
data = clean_keys(data)
scenes = data.get('scenes', data) if isinstance(data, dict) else data

print(f"🎬 Učitano {len(scenes)} scena iz JSON-a.")
print(f"📹 Output mapa: {RAW_VIDEO_DIR}\n")


# ============================================
# DEFINICIJA GENERATE_VIDEO FUNKCIJE (Isi-dev Backend)
# ============================================
def generate_video(positive_prompt, negative_prompt, width, height, seed, steps, cfg_scale, sampler_name, scheduler, frames, fps, output_format, output_filename):
    """
    Generiše video koristeći već inicijalizovane ComfyUI čvorove unutar ove Colab sesije.
    """
    # Povlačenje instanci modela koje si već učitao/la u memoriju u prethodnim ćelijama
    global clip_loader, clip_encode_positive, clip_encode_negative, empty_latent_video, unet_loader, ksampler, vae_loader

    with torch.inference_mode():
        # [KORAK 1] Tekstualno kodiranje prompta
        clip_files = glob.glob("/content/ComfyUI/models/clip/*.safetensors")
        clip_file = os.path.basename(clip_files[0]) if clip_files else "umt5_xxl_fp8_e4m3fn_scaled.safetensors"

        clip = clip_loader.load_clip(clip_file, "wan", "default")[0]
        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]
        del clip
        torch.cuda.empty_cache()
        gc.collect()

        # [KORAK 2] Kreiranje praznog latentnog prostora za video
        empty_latent = empty_latent_video.generate(width, height, frames)[0]

        # [KORAK 3] Učitavanje DiT/UNet modela iz foldera i uzorkovanje (Sampling)
        unet_files = glob.glob("/content/ComfyUI/models/unet/*.safetensors")
        unet_file = os.path.basename(unet_files[0]) if unet_files else "wan2.1_14b_t2v_720p_q4_k_m.safetensors"

        model = unet_loader.load_unet(unet_file)[0]

        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive,
            negative=negative,
            latent_image=empty_latent
        )[0]

        del model
        torch.cuda.empty_cache()
        gc.collect()

        # [KORAK 4] Učitavanje VAE modela i dekodiranje u frejmove slika
        vae_files = glob.glob("/content/ComfyUI/models/vae/*.safetensors")
        vae_file = os.path.basename(vae_files[0]) if vae_files else "wan_2.1_vae.safetensors"
        vae = vae_loader.load_vae(vae_file)[0]

        # Dinamička provjera strukture dekodera (zavisno od verzije wrapper-a)
        try:
            if hasattr(vae, "decode"):
                decoded_frames = vae.decode(sampled["samples"])
            else:
                from nodes import VAEDecode
                decoded_frames = VAEDecode().decode(vae, sampled)[0]
        except Exception:
            decoded_frames = vae.decode(sampled)

        del vae
        torch.cuda.empty_cache()
        gc.collect()

        # [KORAK 5] Kompajliranje frejmova u finalni MP4 fajl preko imageio-a
        output_dir = "/content/ComfyUI/output"
        os.makedirs(output_dir, exist_ok=True)
        output_path = f"{output_dir}/{output_filename}.{output_format}"

        frames_np = [(img.cpu().numpy() * 255).astype(np.uint8) for img in decoded_frames]

        with imageio.get_writer(output_path, fps=fps) as writer:
            for frame in frames_np:
                writer.append_data(frame)

        return output_path


# ============================================
# 4. OPTIMIZOVANE GGUF POSTAVKE ZA WAN 2.1 14B
# ============================================
negative_prompt = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走, blurry, watermark, text, low quality, static, morphing, ugly, deformed, noisy, blurry"

width = 832       # Odnos 16:9 optimizovan za VRAM
height = 480      # Generisanje u 480p (upscale raditi naknadno)
steps = 20        # Optimalni broj koraka za Wan 2.1 GGUF
cfg_scale = 3.5   # Važno: GGUF zahtijeva niži CFG (između 3.0 i 5.0)
sampler_name = "euler"
scheduler = "simple"
frames = 94       # Broj frejmova
fps = 7           # Brzina reprodukcije sirovog videa
output_format = "mp4"

# ============================================
# 5. GLAVNA PETLJA ZA BATCH PROCESIRANJE SCENA
# ============================================
for i, scene in enumerate(scenes):
    sid = str(scene.get('id', '')).strip()
    prompt = scene.get('prompt', '')

    # Dodavanje kinematografskog stila na tvoj prompt iz JSON-a
    full_prompt = f"{prompt}, cinematic lighting, high quality, masterpiece, 8k resolution, photorealistic, Project Gemini dark sci-fi style"

    vid_path = f"{RAW_VIDEO_DIR}/scene_{sid}.mp4"

    # Checkpointing: Ako fajl već postoji i veći je od 10KB, preskače se
    if os.path.exists(vid_path) and os.path.getsize(vid_path) > 10000:
        print(f"[{i+1}/{len(scenes)}] ⏩ Scena {sid} već postoji (preskočeno).")
        continue

    print(f"[{i+1}/{len(scenes)}] 🎨 Generišem Scenu {sid}...", end=" ", flush=True)

    try:
        with torch.inference_mode():
            temp_filename = f"scene_{sid}"

            # Poziv kreirane funkcije
            output_path = generate_video(
                positive_prompt=full_prompt,
                negative_prompt=negative_prompt,
                width=width,
                height=height,
                seed=int(sid) if sid.isdigit() else 42,
                steps=steps,
                cfg_scale=cfg_scale,
                sampler_name=sampler_name,
                scheduler=scheduler,
                frames=frames,
                fps=fps,
                output_format=output_format,
                output_filename=temp_filename
            )

            # Premještanje završenog videa na Google Drive mapu
            if output_path and os.path.exists(output_path):
                shutil.move(output_path, vid_path)
                print(f"✅ Gotovo! ({frames}fr @ {fps}fps)")
            else:
                print("⚠️ Greška: Funkcija nije kreirala fajl na privremenoj putanji.")

    except Exception as e:
        print(f"❌ Greška u generisanju: {e}")

    # Čišćenje VRAM-a nakon svake scene radi sprječavanja "Out of Memory" kraha
    if 'clear_memory' in globals():
        clear_memory()
    else:
        torch.cuda.empty_cache()
        gc.collect()

print("\n🎉 Batch generisanje završeno! Svi raw videi su u mapi 03_raw_video.")

🎬 Učitano 441 scena iz JSON-a.
📹 Output mapa: /content/drive/MyDrive/Andromeda_Hunter_4K/03_raw_video

[1/441] 🎨 Generišem Scenu 000... ❌ Greška: name 'generate_video' is not defined
[2/441] 🎨 Generišem Scenu 001... ❌ Greška: name 'generate_video' is not defined
[3/441] 🎨 Generišem Scenu 002... ❌ Greška: name 'generate_video' is not defined
[4/441] 🎨 Generišem Scenu 003... ❌ Greška: name 'generate_video' is not defined
[5/441] 🎨 Generišem Scenu 004... ❌ Greška: name 'generate_video' is not defined
[6/441] 🎨 Generišem Scenu 005... ❌ Greška: name 'generate_video' is not defined
[7/441] 🎨 Generišem Scenu 006... ❌ Greška: name 'generate_video' is not defined
[8/441] 🎨 Generišem Scenu 007... ❌ Greška: name 'generate_video' is not defined
[9/441] 🎨 Generišem Scenu 008... ❌ Greška: name 'generate_video' is not defined
[10/441] 🎨 Generišem Scenu 009... ❌ Greška: name 'generate_video' is not defined
[11/441] 🎨 Generišem Scenu 010... ❌ Greška: name 'generate_video' is not defined
[12/441] 🎨 Gene

KeyboardInterrupt: 